# Baseline accuracy of UCSF-PDGM using Logistic Regression

In [1]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from methods.logistic_flatten import batch_flatten_pixels_parallel

## Load and clean dataset

In [2]:
ds = load_dataset("chehablab/UCSF_PDGM", split="train")
df = pd.DataFrame(ds, columns=["volume_id", "slice_id", "t1", "t1c", "t2", "is_tumorous", "sex", "age"])
df = df.dropna(subset=["sex"]) # Remove all empty metadata columns
df.sex = (df.sex == "M").astype(np.float32)

## Split dataset into pixel and metadata as x training and test sets and is_tumourous as y

In [3]:
# Split the dataset
patient_ids = np.asarray(df["volume_id"].unique(), dtype=object) # Splits by patient
train_ids, test_ids = train_test_split(patient_ids, test_size=0.3, random_state=1606009)
train_df = df[df["volume_id"].isin(train_ids)]
train_df = train_df.reset_index(drop=True)
test_df = df[df["volume_id"].isin(test_ids)]
test_df = test_df.reset_index(drop=True)

In [4]:
# Split into x for metadata and pixels and y sets
metadata = ["slice_id", "sex", "age"]
pixel = ["t1", "t1c", "t2"]

y_train = np.asarray(train_df["is_tumorous"])
y_test = np.asarray(test_df["is_tumorous"])
x_train_meta = train_df[metadata]
x_train_pixel = train_df[pixel]
x_test_meta = test_df[metadata]
x_test_pixel = test_df[pixel]

## Logistic Regression on metadata to predict is_tumourous

In [5]:
# Metadata logistic regression
meta_scaler = StandardScaler()
x_train_meta_scaled = meta_scaler.fit_transform(x_train_meta)
x_test_meta_scaled = meta_scaler.fit_transform(x_test_meta)

logreg = LogisticRegression(max_iter=1000, random_state=1606009)
logreg.fit(x_train_meta_scaled, y_train)
meta_pred = logreg.predict(x_test_meta_scaled)
meta_proba = logreg.predict_proba(x_test_meta_scaled)[:, 1]

print("Baseline metadata logistic regression")
print(classification_report(y_test, meta_pred, target_names=["No tumour", "Tumourous"]))
print(f"ROC-AUC: {roc_auc_score(y_test, meta_proba):.4f}")

Baseline metadata logistic regression
              precision    recall  f1-score   support

   No tumour       0.62      0.73      0.67     11587
   Tumourous       0.45      0.33      0.38      7788

    accuracy                           0.57     19375
   macro avg       0.54      0.53      0.53     19375
weighted avg       0.55      0.57      0.55     19375

ROC-AUC: 0.6916


## Logistic Regression on pixels to predict is_tumourous

In [6]:
# Pixels logistic regression
print("Flattening images to 32*32*3 (T1, T1c, T2) for pixel baseline...")
x_train_pixel_flat = batch_flatten_pixels_parallel(x_train_pixel)   # In logistic_flatten.py
x_test_pixel_flat = batch_flatten_pixels_parallel(x_test_pixel)

Flattening images to 32*32*3 (T1, T1c, T2) for pixel baseline...
Pre-allocating matrix for 44795 records...
Preparing image streams for multiprocessing...
Launching parallel processing...
 -> Progress: 10000/44795 rows compiled successfully.
 -> Progress: 20000/44795 rows compiled successfully.
 -> Progress: 30000/44795 rows compiled successfully.
 -> Progress: 40000/44795 rows compiled successfully.
 -> Progress: 44795/44795 rows compiled successfully.
Normalization across the whole dataset...
Final matrix shape: (44795, 3072)
Pre-allocating matrix for 19375 records...
Preparing image streams for multiprocessing...
Launching parallel processing...
 -> Progress: 10000/19375 rows compiled successfully.
 -> Progress: 19375/19375 rows compiled successfully.
Normalization across the whole dataset...
Final matrix shape: (19375, 3072)


In [7]:
logreg = LogisticRegression(max_iter=1000, random_state=1606009)
logreg.fit(x_train_pixel_flat, y_train)
pixel_pred = logreg.predict(x_test_pixel_flat)
pixel_proba = logreg.predict_proba(x_test_pixel_flat)[:, 1]

print("Baseline pixel logistic regression")
print(classification_report(y_test, pixel_pred, target_names=["No tumour", "Tumourous"]))
print(f"ROC-AUC: {roc_auc_score(y_test, pixel_proba):.4f}")

Baseline pixel logistic regression
              precision    recall  f1-score   support

   No tumour       0.80      0.82      0.81     11587
   Tumourous       0.72      0.68      0.70      7788

    accuracy                           0.77     19375
   macro avg       0.76      0.75      0.76     19375
weighted avg       0.77      0.77      0.77     19375

ROC-AUC: 0.8583


## Logistic Regression on metadata and pixels to predict is_tumourous

In [8]:
# Combined logistic regression
x_train = np.hstack([x_train_meta_scaled, x_train_pixel_flat])
x_test = np.hstack([x_test_meta_scaled, x_test_pixel_flat])
logreg = LogisticRegression(max_iter=1000, random_state=1606009)
logreg.fit(x_train, y_train)
pred = logreg.predict(x_test)
proba = logreg.predict_proba(x_test)[:, 1]

print("Baseline combined pixel + metadata logistic regression")
print(classification_report(y_test, pred, target_names=["No tumour", "Tumourous"]))
print(f"ROC-AUC: {roc_auc_score(y_test, proba):.4f}")

Baseline combined pixel + metadata logistic regression
              precision    recall  f1-score   support

   No tumour       0.80      0.83      0.81     11587
   Tumourous       0.73      0.69      0.71      7788

    accuracy                           0.77     19375
   macro avg       0.76      0.76      0.76     19375
weighted avg       0.77      0.77      0.77     19375

ROC-AUC: 0.8599
